# Additional analysis: probing the properties of PSEs

In this analysis, we show that PSEs can reliably predict some properties of biological sequences. To do so, we train a `Ridge` regressor (tuning its regularization hyperparameter `alpha` with `RandomizedSearchCV`) with 5x5 nested cross validation to predict 8 different properties of sequences.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.model_selection import cross_validate, RandomizedSearchCV
from sklearn.linear_model import Ridge

df_pos = pd.read_parquet("../data/pses/lobo/training/positive.parquet")
df_neg = pd.read_parquet("../data/pses/lobo/training/negative.parquet")
df = pd.concat([df_pos, df_neg], axis=0, ignore_index=True)

column_indices = slice(7, df.shape[1])

In [80]:
from Bio.Seq import Seq
from Bio.SeqUtils.ProtParam import ProteinAnalysis

targets = [
    "aromaticity",
    "gravy",
    "instability_index",
    "isoelectric_point",
    "length",
    "molar_extinction_coefficient",
    "molecular_weight",
    "secondary_structure_fraction",
]


def get_target(name, sequences):
    targets = []
    for raw_seq in sequences:
        seq = Seq(raw_seq)
        analysis = ProteinAnalysis(seq)
        prop = getattr(analysis, name)
        if name != "length":
            prop = prop()
        targets.append(prop)
    return np.array(targets)

In [81]:
np.random.seed(0)
perm = np.random.permutation(df.shape[0])

for target_name in targets:
    X = df.iloc[:, column_indices]
    y = get_target(target_name, df.Seq.tolist())
    model = RandomizedSearchCV(Ridge(), {"alpha": stats.loguniform(0.1, 1000)}, n_iter=50)
    results = cross_validate(model, X.values[perm], y[perm], n_jobs=-1)
    m, s = results["test_score"].mean(), results["test_score"].std()
    print(target_name, f"{m:.3f} +/- {s:.3f}")

aromaticity 0.902 +/- 0.034
gravy 0.956 +/- 0.004
instability_index 0.540 +/- 0.086
isoelectric_point 0.735 +/- 0.032
length 0.682 +/- 0.046
molar_extinction_coefficient 0.660 +/- 0.038
molecular_weight 0.679 +/- 0.048
secondary_structure_fraction 0.929 +/- 0.007
